In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'

In [ ]:
liftover = pd.read_csv(f'{projdir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', index_col=0)
liftover.shape

In [ ]:
liftover.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome'
multiome_clusters = pd.read_csv(f'{indir}/csv/clusters/rna_leiden_v7.csv', sep='\t', index_col=0)
multiome_clusters.shape

In [ ]:
multiome_clusters.head()

In [ ]:
indir = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v6_Data'
name = 'YS3v6.NameCluster.DiffExp.Table.Big.txt'
degs = pd.read_csv(f'{indir}/{name}', sep='\t')
degs['coarse'] = [x.split('_')[0] for x in degs['cluster']]
degs.shape

In [ ]:
degs.head()

In [ ]:
multiome = sc.read_h5ad(f'{projdir}/h5ad/raw/multiome.passqc_nuclei.h5ad')
snmct = sc.read_h5ad(f'{projdir}/h5ad/raw/mct.passqc_nuclei.h5ad')

In [ ]:
multiome.X[:100,:100].toarray()

In [ ]:
snmct.X[:100,:100].toarray()

In [ ]:
sc.pp.normalize_total(multiome, target_sum=1e4)
sc.pp.normalize_total(snmct, target_sum=1e4)

In [ ]:
sc.pp.log1p(multiome)
sc.pp.log1p(snmct)

In [ ]:
snmct.obs['cluster'] = liftover.reindex(snmct.obs.index)['pred_coarse']
multiome.obs['cluster'] = multiome_clusters.reindex(multiome.obs.index)['coarse_v7']

In [ ]:
snmct.obs.head()

In [ ]:
multiome.obs.head()

In [ ]:
clusts = degs['coarse'].unique()

snmct_results = []
multiome_results = []

for s in clusts:
    tmp_snmct = snmct[snmct.obs['cluster'] == s].X.mean(axis=0)
    tmp_multiome = multiome[multiome.obs['cluster'] == s].X.mean(axis=0)
    snmct_results.append(np.ravel(tmp_snmct))
    multiome_results.append(np.ravel(tmp_multiome))

In [ ]:
multiome_df = pd.DataFrame(multiome_results, index=clusts)
multiome_df.index = [f'multiome_{x}' for x in multiome_df.index]

snmct_df = pd.DataFrame(snmct_results, index=clusts)
snmct_df.index = [f'snmct_{x}' for x in snmct_df.index]

merged_df = pd.concat([multiome_df, snmct_df], axis=0)

In [ ]:
corr_df = merged_df.T.corr()
corr_df.shape

In [ ]:
corr_df.head()

In [ ]:
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']
tmp_multiome = [f'multiome_{x}' for x in tmp_order]
tmp_snmct = [f'snmct_{x}' for x in tmp_order]
reorder = tmp_multiome + tmp_snmct
final_corr = corr_df.loc[reorder,reorder]
final_corr.index = [x.split('_')[1] for x in final_corr.index]
final_corr.columns = [x.split('_')[1] for x in final_corr.columns]

In [ ]:
fig, axes = plt.subplots(1, figsize=(12, 6))
sns.heatmap(final_corr, vmin=0.5, vmax=1, cmap='coolwarm')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
final_corr.to_csv(f'{outdir}/multiome_snmct_pseudo_corr.csv', sep='\t', header=True, index=True)

In [ ]:
!date